# Weather Data

In [2]:
import cdsapi # important in order to use the CDS API
import pandas as pd
import numpy as np
import zipfile
import os
from glob import glob
import holidays

## CDS API Call 

In [3]:
# config dictionary

CONFIG = {
    # variables we are interested in
    "variables": [
        "2m_temperature", 
        "total_precipitation",
        "snow_cover",
        "snow_depth",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind"
    ],
    "date": "2024-01-01/2026-05-12",
    "dir_name": "era5_data.zip",
}

In [4]:
# API call of the CDS API which can be generated on https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=download

dataset = "reanalysis-era5-land-timeseries"

request = {
    "variable": CONFIG["variables"],
    "location": {"longitude": -87.8, "latitude": 42}, # coordinates of Chicago
    "date": CONFIG["date"],
    "data_format": "csv"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download(CONFIG["dir_name"])

2026-06-16 14:01:53,600 INFO [2026-02-16T00:00:00] - To generate this ERA5-land hourly time series dataset, **homogenisation conventions have been applied to the ERA5 source GRIB data** to ensure consistency, usability, and alignment across chosen variables and time steps. The processed data were then written to an **ARCO Zarr archive**, enabling efficient cloud-optimised access and scalable data retrieval. Please refer to the [user guide](https://confluence.ecmwf.int/x/Dg32Hw) for details.

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=overview). **Requirements for additional parameters may be considered**. Please raise your request with ECMWF Support [here](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202).
2026-06-16 14:01:53,601 INFO Request ID is 82861d83-c26f-4b12-a01c-b0a164afd383
2026-06-16 14:01:53,680 INFO st

2fa06f29843a4e528fa2ffc7409d12c9.zip:   0%|          | 0.00/652k [00:00<?, ?B/s]

'era5_data.zip'

## Load individual CSV files 

In [5]:
with zipfile.ZipFile(CONFIG["dir_name"], "r") as zip_ref:
    zip_ref.extractall("era5_data")

print(os.listdir("era5_data"))

['reanalysis-era5-land-timeseries-sfc-2m-temperature8ewbdqyd.csv', 'reanalysis-era5-land-timeseries-sfc-pressure-precipitationa68zhuxz.csv', 'reanalysis-era5-land-timeseries-sfc-snownhln6th9.csv', 'reanalysis-era5-land-timeseries-sfc-windpepkitg9.csv']


In [6]:
# Alle CSV-Dateien finden
files = glob("era5_data/*.csv")

# Dictionary für die DataFrames
dfs = {}

for file in files:
    filename = os.path.basename(file)

    # Feature aus Dateiname ableiten
    if "temperature" in filename:
        key = "temp"
    elif "precipitation" in filename:
        key = "precip"
    elif "snow" in filename:
        key = "snow"
    elif "wind" in filename:
        key = "wind"

    # CSV laden
    dfs[key] = pd.read_csv(file, encoding="latin1")

    print(f"\n--- {key} ---")
    print(dfs[key].head())


--- temp ---
            valid_time        t2m  latitude  longitude
0  2024-01-01 00:00:00  274.39615      42.0      -87.8
1  2024-01-01 01:00:00  274.24170      42.0      -87.8
2  2024-01-01 02:00:00  274.05762      42.0      -87.8
3  2024-01-01 03:00:00  273.90906      42.0      -87.8
4  2024-01-01 04:00:00  274.01672      42.0      -87.8

--- precip ---
            valid_time        tp  latitude  longitude
0  2024-01-01 00:00:00  0.000281      42.0      -87.8
1  2024-01-01 01:00:00  0.000150      42.0      -87.8
2  2024-01-01 02:00:00  0.000030      42.0      -87.8
3  2024-01-01 03:00:00  0.000014      42.0      -87.8
4  2024-01-01 04:00:00  0.000037      42.0      -87.8

--- snow ---
            valid_time     snowc       sde  latitude  longitude
0  2024-01-01 00:00:00  6.929688  0.007812      42.0      -87.8
1  2024-01-01 01:00:00  8.179688  0.008789      42.0      -87.8
2  2024-01-01 02:00:00  8.804688  0.008789      42.0      -87.8
3  2024-01-01 03:00:00  8.873047  0.008789    

## Merge individual Dataframes

In [7]:
df_merged = dfs["temp"].copy()

df_merged = df_merged.merge(
    dfs["precip"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)
df_merged = df_merged.merge(
    dfs["snow"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)
df_merged = df_merged.merge(
    dfs["wind"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)

df_merged

,valid_time,t2m,latitude,longitude,tp,snowc,sde,u10,v10
0,2024-01-01 00:00:00,274.39615,42.0,-87.8,0.000281,6.929688,7.812500e-03,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,42.0,-87.8,0.000150,8.179688,8.789062e-03,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,42.0,-87.8,0.000030,8.804688,8.789062e-03,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,42.0,-87.8,0.000014,8.873047,8.789062e-03,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,42.0,-87.8,0.000037,8.890625,8.789062e-03,2.084305,-6.889282
...,...,...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,42.0,-87.8,0.000342,0.000000,-7.345365e-24,1.670914,4.648880
20708,2026-05-12 20:00:00,293.64343,42.0,-87.8,0.000003,0.000000,-7.345365e-24,1.859116,3.183975
20709,2026-05-12 21:00:00,293.37470,42.0,-87.8,0.000053,0.000000,-7.345365e-24,2.806747,3.596573
20710,2026-05-12 22:00:00,292.62510,42.0,-87.8,0.000749,0.000000,-7.345365e-24,3.238815,4.192703


## Data Preprocessing

In [8]:
# check for missing values
print("--- Missing values ---")
print(df_merged.isna().sum())

# check for data types
print("--- Data types ---")
print(df_merged.dtypes)

# short statistical description of the data
print("--- Stat description ---")
print(df_merged.describe())

--- Missing values ---
valid_time    0
t2m           0
latitude      0
longitude     0
tp            0
snowc         0
sde           0
u10           0
v10           0
dtype: int64
--- Data types ---
valid_time     object
t2m           float64
latitude      float64
longitude     float64
tp            float64
snowc         float64
sde           float64
u10           float64
v10           float64
dtype: object
--- Stat description ---
                t2m      latitude     longitude            tp         snowc  \
count  20712.000000  2.071200e+04  2.071200e+04  2.071200e+04  20712.000000   
mean     283.151036  4.200000e+01 -8.780000e+01  1.109412e-04     10.311244   
std       10.784377  1.136896e-12  2.651810e-11  5.408161e-04     25.109695   
min      248.497120  4.200000e+01 -8.780000e+01 -3.736932e-08      0.000000   
25%      274.883240  4.200000e+01 -8.780000e+01  0.000000e+00      0.000000   
50%      283.333980  4.200000e+01 -8.780000e+01  0.000000e+00      0.000000   
75%      29

### Clean up

In [9]:
df_merged = df_merged.drop(['latitude', 'longitude'], axis=1) # not necessary as always the same location/coordinates for all data points

df_merged = df_merged.rename(
    columns={
        'valid_time': 'time_step',
        't2m': '2m_temp',
        'tp': 'total_precip',
        'snowc': 'snow_cov',
        'sde': 'snow_depth'
    }
)
df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,7.812500e-03,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,8.789062e-03,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,8.789062e-03,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,8.789062e-03,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,8.789062e-03,2.084305,-6.889282
...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,-7.345365e-24,1.670914,4.648880
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,-7.345365e-24,1.859116,3.183975
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,-7.345365e-24,2.806747,3.596573
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,-7.345365e-24,3.238815,4.192703


In [10]:
# change dtype to datetime object
df_merged['time_step'] = pd.to_datetime(df_merged['time_step'])

# remove values that are physically impossible (porbably due to small floating point numbers)
df_merged['total_precip'] = df_merged['total_precip'].clip(lower=0)
df_merged['snow_depth'] = df_merged['snow_depth'].clip(lower=0)

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282
...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,0.000000,1.670914,4.648880
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,0.000000,1.859116,3.183975
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,0.000000,2.806747,3.596573
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,0.000000,3.238815,4.192703


### Conversion of units

In [11]:
# convert temperature from Kelvin to Celsius via formula C = K - 273.15
df_merged['2m_temp_c'] = df_merged['2m_temp'] - 273.15

# convert from m to mm (common unit for precipitation)
df_merged['total_precip_mm'] = df_merged['total_precip'] * 1000

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687,1.24615,0.280723
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841,1.09170,0.150489
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237,0.90762,0.030188
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008,0.75906,0.013527
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282,0.86672,0.037491
...,...,...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,0.000000,1.670914,4.648880,20.15435,0.341564
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,0.000000,1.859116,3.183975,20.49343,0.002533
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,0.000000,2.806747,3.596573,20.22470,0.052869
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,0.000000,3.238815,4.192703,19.47510,0.748754


### Feature engineering

In [12]:
# combine the raw wind data into the wind speed and direction
df_merged["wind_speed"] = (df_merged["u10"]**2 + df_merged["v10"]**2) ** 0.5
df_merged["wind_dir"] = (270 - np.degrees(np.arctan2(df_merged["u10"], df_merged["v10"]))*180/np.pi)%360

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm,wind_speed,wind_dir
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687,1.24615,0.280723,6.555811,353.058005
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841,1.09170,0.150489,7.522287,142.581272
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237,0.90762,0.030188,7.434575,72.573111
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008,0.75906,0.013527,7.282913,79.642464
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282,0.86672,0.037491,7.197676,281.209740
...,...,...,...,...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,0.000000,1.670914,4.648880,20.15435,0.341564,4.940044,217.285548
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,0.000000,1.859116,3.183975,20.49343,0.002533,3.687005,335.052008
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,0.000000,2.806747,3.596573,20.22470,0.052869,4.562145,254.576762
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,0.000000,3.238815,4.192703,19.47510,0.748754,5.297989,270.769812


In [13]:
# split the date into several parts
df_merged['hour'] = df_merged['time_step'].dt.hour
df_merged['day'] = df_merged['time_step'].dt.day
df_merged['month'] = df_merged['time_step'].dt.month
df_merged['weekday'] = df_merged['time_step'].dt.weekday

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm,wind_speed,wind_dir,hour,day,month,weekday
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687,1.24615,0.280723,6.555811,353.058005,0,1,1,0
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841,1.09170,0.150489,7.522287,142.581272,1,1,1,0
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237,0.90762,0.030188,7.434575,72.573111,2,1,1,0
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008,0.75906,0.013527,7.282913,79.642464,3,1,1,0
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282,0.86672,0.037491,7.197676,281.209740,4,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,0.000000,1.670914,4.648880,20.15435,0.341564,4.940044,217.285548,19,12,5,1
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,0.000000,1.859116,3.183975,20.49343,0.002533,3.687005,335.052008,20,12,5,1
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,0.000000,2.806747,3.596573,20.22470,0.052869,4.562145,254.576762,21,12,5,1
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,0.000000,3.238815,4.192703,19.47510,0.748754,5.297989,270.769812,22,12,5,1


In [14]:
df_merged['hour_sin'] = np.sin(2 * np.pi * df_merged['hour'] / 24)
df_merged['hour_cos'] = np.cos(2 * np.pi * df_merged['hour'] / 24)

df_merged['month_sin'] = np.sin(2 * np.pi * df_merged['month'] / 12)
df_merged['month_cos'] = np.cos(2 * np.pi * df_merged['month'] / 12)

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm,wind_speed,wind_dir,hour,day,month,weekday,hour_sin,hour_cos,month_sin,month_cos
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687,1.24615,0.280723,6.555811,353.058005,0,1,1,0,0.000000,1.000000,0.5,0.866025
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841,1.09170,0.150489,7.522287,142.581272,1,1,1,0,0.258819,0.965926,0.5,0.866025
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237,0.90762,0.030188,7.434575,72.573111,2,1,1,0,0.500000,0.866025,0.5,0.866025
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008,0.75906,0.013527,7.282913,79.642464,3,1,1,0,0.707107,0.707107,0.5,0.866025
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282,0.86672,0.037491,7.197676,281.209740,4,1,1,0,0.866025,0.500000,0.5,0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,0.000000,1.670914,4.648880,20.15435,0.341564,4.940044,217.285548,19,12,5,1,-0.965926,0.258819,0.5,-0.866025
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,0.000000,1.859116,3.183975,20.49343,0.002533,3.687005,335.052008,20,12,5,1,-0.866025,0.500000,0.5,-0.866025
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,0.000000,2.806747,3.596573,20.22470,0.052869,4.562145,254.576762,21,12,5,1,-0.707107,0.707107,0.5,-0.866025
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,0.000000,3.238815,4.192703,19.47510,0.748754,5.297989,270.769812,22,12,5,1,-0.500000,0.866025,0.5,-0.866025


In [15]:
# introduce holidays as feature
us_holidays = holidays.US(state="IL")

df_merged["date"] = df_merged["time_step"].dt.date
df_merged["is_holiday"] = df_merged["date"].isin(us_holidays).astype(int)

# also flag dates that are close to a holiday
df_merged["is_near_holiday"] = (
    df_merged["date"].isin(us_holidays) |
    df_merged["date"].shift(1).isin(us_holidays) |
    df_merged["date"].shift(-1).isin(us_holidays)
).astype(int)

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm,wind_speed,...,day,month,weekday,hour_sin,hour_cos,month_sin,month_cos,date,is_holiday,is_near_holiday
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687,1.24615,0.280723,6.555811,...,1,1,0,0.000000,1.000000,0.5,0.866025,2024-01-01,0,0
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841,1.09170,0.150489,7.522287,...,1,1,0,0.258819,0.965926,0.5,0.866025,2024-01-01,0,0
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237,0.90762,0.030188,7.434575,...,1,1,0,0.500000,0.866025,0.5,0.866025,2024-01-01,0,0
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008,0.75906,0.013527,7.282913,...,1,1,0,0.707107,0.707107,0.5,0.866025,2024-01-01,0,0
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282,0.86672,0.037491,7.197676,...,1,1,0,0.866025,0.500000,0.5,0.866025,2024-01-01,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,0.000000,1.670914,4.648880,20.15435,0.341564,4.940044,...,12,5,1,-0.965926,0.258819,0.5,-0.866025,2026-05-12,0,0
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,0.000000,1.859116,3.183975,20.49343,0.002533,3.687005,...,12,5,1,-0.866025,0.500000,0.5,-0.866025,2026-05-12,0,0
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,0.000000,2.806747,3.596573,20.22470,0.052869,4.562145,...,12,5,1,-0.707107,0.707107,0.5,-0.866025,2026-05-12,0,0
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,0.000000,3.238815,4.192703,19.47510,0.748754,5.297989,...,12,5,1,-0.500000,0.866025,0.5,-0.866025,2026-05-12,0,0


## Data Export
Export the data into its own CSV file in order to merge it with POI and taxi data later on. 

In [17]:
# Export weather data table to use in other notebooks
df_merged.to_parquet("../data/weather_data.parquet")